In [1]:
import sys
sys.path.append("..")  
from helpers import *
import pandas as pd
import csv, os
from datetime import datetime


In [2]:
def log_feedback(user_id, movie_title, action, reason="", genre=""):
    """Insert one feedback event into Supabase."""
    try:
        supabase.table("feedback").insert({
            "user_id": str(user_id),
            "movie_title": movie_title,
            "action": action,
            "reason": reason,
            "genre": genre,
            "timestamp": datetime.now().isoformat()
        }).execute()
    except Exception as e:
        print(f"Feedback log failed: {e}")     # don't crash the app

In [3]:
log_feedback("test", "Test Movie", "reject", reason="storyline", genre="Drama")

In [6]:
def get_user_feedback(user_id):
    """Read a user's rejections from Supabase."""
    try:
        res = supabase.table("feedback").select("*") \
            .eq("user_id", str(user_id)) \
            .eq("action", "reject").execute()
        rows = res.data
    except Exception as e:
        print(f"Feedback read failed: {e}")
        return set(), set()

    rejected_titles = {r["movie_title"] for r in rows}
    avoided_genres = {r["genre"] for r in rows
                      if r["reason"] == "genre_mismatch" and r["genre"]}
    return rejected_titles, avoided_genres

In [18]:
print(get_user_feedback(1))   # should print ({'Inception', 'Top Gun'}, {'Sci-Fi'})

({'Top Gun', 'Inception'}, {'Sci-Fi'})


In [ ]:
def filter_rejected(movies, rejected_titles):
    """
    Remove movies the user has rejected.
    Works with BOTH dicts (Guest) and title strings (Personal).
    """
    if not rejected_titles:
        return movies

    kept = []
    for m in movies:
        title = m["title"] if isinstance(m, dict) else m     # handle both shapes 
        if title not in rejected_titles:
            kept.append(m)
    return kept